### __Goals__
- To dive into `human-in-the-loop` which allows the users to interact with the graph directly in various ways
- Understand `streaming` which provides severals ways to visualize graph output(e.g node state or chat model token usage) over the the course of execution
- `Approval`
- `Debugging`
- `Editing`

In [ ]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join("..")))
from IPython.display import Image, display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from utils.model import Models


In [ ]:
model = Models()
llm = model.create_groq_model()

#### __Breakpoints for human approval__
- Compile the graph with `interrupt_before=['tools']` where `tools` is the node before which we want to interact with the graph

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply the two arguments and return the result
    Args:
    a:First Argument
    b:Second Argument
    """
    return a * b


def addition(a: int, b: int) -> int:
    """Add the two arguments and return the result
    Args:
    a:First Argument
    b:Second Argument
    """
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract the two arguments and return the result
    Args:
    a:First Argument
    b:Second Argument
    """
    return a - b


In [ ]:
tools = [multiply, addition, subtract]


In [ ]:
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def assistant(state: MessagesState):
    sys_msg = SystemMessage(
        content="You are a helpful assistant tasked to do only arithmetic operations"
    )
    return {"messages": llm_with_tools.invoke([sys_msg] + state["messages"])}

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")
memory = MemorySaver()
graph = builder.compile(interrupt_before=["tools"], checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
input_config: RunnableConfig = {"configurable": {"thread_id": "1"}}

In [ ]:
initial_input = {"messages": "multiply 2 and 3"}

In [ ]:
for event in graph.stream(initial_input, input_config, stream_mode="values"):
    event["messages"][-1].pretty_print()

In [ ]:
state = graph.get_state(input_config)
state.next

In [ ]:
state

If we call the graph.stream() method with `None` and the same `config` and `thread_id`, the graph resumes execution from the node where it was interrupted 

In [ ]:
for event in graph.stream(None, input_config, stream_mode="values"):
    event["messages"][-1].pretty_print()

#### Human in the loop 

In [ ]:
input_config_1: RunnableConfig = {"configurable": {"thread_id": "2"}}
input_msg = HumanMessage("multiply 3 and 5")


In [ ]:
for event in graph.stream(
    {"messages": [input_msg]}, config=input_config_1, stream_mode="values"
):
    event["messages"][-1].pretty_print()

user_input = input("Do you want to continue? Yes or NO")
if user_input.lower() == "yes":
    for event in graph.stream(None, input_config_1, stream_mode="values"):
        event["messages"][-1].pretty_print()
else:
    print("Operation cancelled")